# MoS1aD
## Dependencies

In [1]:
import os
import tqdm
import shutil
import numpy as np
import scipy.io
import pandas as pd
import pickle
import h5py

import logging

In [3]:
def load_mocap_data(file_path):
    mat = scipy.io.loadmat(file_path)
    exp_name = [m for m in mat.keys() if m[:2] != '__'][0]  ## TOCHANGE
    data = np.moveaxis(mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Data'][0, 0], 2, 0)
    keypoints = [label[0].replace('coordinate', 'coord') for label in
                    mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Labels'][0, 0][0]]

    # Very important. Make sure the keypoints are always in the same order even if not saved so in the original files
    new_order = np.argsort(keypoints)
    keypoints = [keypoints[n] for n in new_order]
    data = data[:, new_order, :]

    return data, keypoints

## CLB

In [6]:
file_path = "/home/group-cvg/cvg-rose/bogna_mocap/MoS1aD_CLB"
files = os.listdir(file_path)
files.remove("MOS1aD_S12_M9_MC3_T3_CLB_2023_04_11_processed_THT_2024_01_29_bij_B.mat") # can not load
print(len(files))
files

24


['MOS1aD_S8_M4_MC2_T1_CLB_2023_04_11_processed_THT_2024_01_27_bij_B.mat',
 'MOS1aD_S13_M10_MC3_T4_CLB_2023_04_11_processed_THT_2024_02_01_bij_A2.mat',
 'MOS1aD_S10_M6_MC2_T3_CLB_2023_04_11_processed_THT_2024_01_29_bij_E.mat',
 'MOS1aD_S9_M5_MC2_T2_CLB_2023_04_11_processed_THT_2024_01_27_bij_A1.mat',
 'MOS1aD_S22_M6_MC2_T3_CLB_2023_04_18_processed_THT_2024_02_02_bij_B.mat',
 'MOS1aD_S1_M4_MC2_T1_CLB_2023_04_07_processed_THT_2024_01_23_bij_A2.mat',
 'MOS1aD_S33_M5_MC2_T2_CLB_2023_04_25_processed_THT_2024_02_26_bij_A2.mat',
 'MOS1aD_S15_M5_MC2_T2_CLB_2023_04_14_processed_THT_2024_02_01_bij_B.mat',
 'MOS1aD_S29_M8_MC3_T2_CLB_2023_04_21_processed_THT_2024_02_03_bij_A2.mat',
 'MOS1aD_S5_M8_MC3_T2_CLB_2023_04_07_processed_THT_2024_01_31_bij_B.mat',
 'MOS1aD_S34_M5_MC2_T3_CLB_2023_04_25_processed_THT_2024_02_28_bij_A2.mat',
 'MOS1aD_S35_M8_MC3_T2_CLB_2023_04_25_processed_THT_2024_03_29_bij_A1.mat',
 'MOS1aD_S6_M9_MC3_T3_CLB_2023_04_07_processed_THT_2024_01_25_bij_A1.mat',
 'MOS1aD_S7_M10_MC3_T

In [7]:
data_MOS1aD_CLB = {}

for file_name in files:
    file = os.path.join(file_path, file_name)
    
    if os.path.isfile(file):
        data, keypoints = load_mocap_data(file)
        key_name = str(file_name.split("_")[0] + "_" + file_name.split("_")[1]) + "_CLB"
        data_MOS1aD_CLB[key_name] ={"label":None, "keypoints":None}
        data_MOS1aD_CLB[key_name]["label"] = file_name.split(".")[-2][-1]
        data_MOS1aD_CLB[key_name]["keypoints"] = keypoints
        # 1. Discard last dimension
        data = data[ : , : , :3]
        
        data = data[1500: ]
        
        data_MOS1aD_CLB[key_name]["data"] = data

In [8]:
# The correct keypoint 
keypoints_name = ['left_ankle',  'left_back',  'left_coord',  'left_hip',  'left_knee', 
                  'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']

for key, value in data_MOS1aD_CLB.items():
    # Delete miniscope keypoints
    index = [i for i, s in enumerate(value["keypoints"]) if "miniscope" in s]
    if len(index)> 0:
        value["keypoints"].pop(index[0]) 
        value["data"] = np.delete(value["data"], index[0], axis=1)

    # Create an empty array to store
    arr = np.full((18000, 10, 3), np.nan)
    i = 0  # index of correct keypoint
    j = 0  # index of actual keypoint
    while i < 10:
        if keypoints_name[i] == value["keypoints"][i]:
            arr[:, i, :] = value["data"][:, j, :]
            i += 1 
            j += 1
        else:
            value["keypoints"].insert(i, None)
            i += 1
    
    value["data"] = arr

In [9]:
for key, value in data_MOS1aD_CLB.items():
    print(key, value["data"].shape, value["label"], value["keypoints"])

MOS1aD_S8_CLB (18000, 10, 3) B ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']
MOS1aD_S13_CLB (18000, 10, 3) 2 ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']
MOS1aD_S10_CLB (18000, 10, 3) E ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']
MOS1aD_S9_CLB (18000, 10, 3) 1 ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']
MOS1aD_S22_CLB (18000, 10, 3) B ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']
MOS1aD_S1_CLB (18000, 10, 3) 2 ['left_ankle', 'left_back', 'left_coord', 'left_hip', 'left_knee', 'right_ankle', 'right_back', 'right_coord', 'right_hip

In [10]:
import pickle

with open('../../data/MOS1aD_CLB.pkl', 'wb') as file:
    pickle.dump(data_MOS1aD_CLB, file)

## Check data

In [ ]:
import pickle

# Read data
with open('/home/rguo_hpc/myfolder/code/mocap/data/MOS1aD_CLB.pkl', 'rb') as file:
    data = pickle.load(file)